In [86]:
import polars as pl
import numpy as np
PARQUET_PATH = r"bts_all.parquet"
COLS = ["timestamp", "item_id", "position", "click", "propensity_score"]

In [87]:
df = (
    pl.read_parquet(PARQUET_PATH, columns=COLS)
      .with_columns(
          pl.col("timestamp").str.to_datetime(time_zone="UTC").alias("ts")
      )
)

print("Shape:", df.shape)
print("Columns:", df.columns)
print("Types:")
display(df.schema)
print("\nNulls:")
display(df.null_count())
print("\nHead:")
display(df.head())

Shape: (12357200, 6)
Columns: ['timestamp', 'item_id', 'position', 'click', 'propensity_score', 'ts']
Types:


Schema([('timestamp', String),
        ('item_id', Int64),
        ('position', Int64),
        ('click', Int64),
        ('propensity_score', Float64),
        ('ts', Datetime(time_unit='us', time_zone='UTC'))])


Nulls:


timestamp,item_id,position,click,propensity_score,ts
u32,u32,u32,u32,u32,u32
0,0,0,0,0,0



Head:


timestamp,item_id,position,click,propensity_score,ts
str,i64,i64,i64,f64,"datetime[μs, UTC]"
"""2019-11-24 00:00:00.047846+00:…",18,2,0,0.0613,2019-11-24 00:00:00.047846 UTC
"""2019-11-24 00:00:00.049952+00:…",61,1,0,0.104925,2019-11-24 00:00:00.049952 UTC
"""2019-11-24 00:00:00.052473+00:…",51,3,0,0.04862,2019-11-24 00:00:00.052473 UTC
"""2019-11-24 00:00:00.503685+00:…",52,1,0,0.04298,2019-11-24 00:00:00.503685 UTC
"""2019-11-24 00:00:00.505885+00:…",35,2,0,0.00589,2019-11-24 00:00:00.505885 UTC


In [49]:
n_dups = df.height - df.unique().height
print(f"Полных дубликатов: {n_dups:,}")

Полных дубликатов: 2,697


In [50]:

dups_rows = (
    df.group_by(COLS)
      .len()
      .filter(pl.col("len") > 1)
)
print(f"Уникальных строк, которые дублируются: {dups_rows.height}")

total_dup_rows = (dups_rows["len"] - 1).sum()
print(f"Лишних строк из-за дублирования: {total_dup_rows}")

Уникальных строк, которые дублируются: 2295
Лишних строк из-за дублирования: 2697


In [51]:
df_before = df.height
df = df.unique()
print(f"Было: {df_before:,}")
print(f"Стало: {df.height:,}")
print(f"Удалено: {df_before - df.height:,}")

Было: 12,357,200
Стало: 12,354,503
Удалено: 2,697


In [52]:
click_rate = df["click"].mean()
print(f"Click rate: {click_rate:.6f} ({click_rate*100:.4f}%)")
print(f"Всего кликов: {df['click'].sum():,} из {df.height:,}")

print("\nРаспределение по позициям:")
display(
    df.group_by("position")
      .agg(pl.len().alias("total_amount"), pl.col("click").mean().alias("CTR"))
      .sort("position")
)

Click rate: 0.004784 (0.4784%)
Всего кликов: 59,108 из 12,354,503

Распределение по позициям:


position,total_amount,CTR
i64,u32,f64
1,4117512,0.004776
2,4118653,0.004808
3,4118338,0.004769


In [53]:
item_stats = (
    df.group_by("item_id")
      .agg(pl.len().alias("impressions"),
           pl.col("click").mean().alias("CTR"))
      .sort("impressions", descending=True)
)

print("Уникальных item_id:", item_stats.height)
print("\nТоп 10 item_id по показам:")
display(item_stats.head(10))
print("\n10 самых редких item_id:")
display(item_stats.tail(10))

top1_share = item_stats["impressions"][0] / df.height
print(f"\nДоля показов топ 1 item: {top1_share:.4f}")

Уникальных item_id: 80

Топ 10 item_id по показам:


item_id,impressions,CTR
i64,u32,f64
51,1350874,0.00551
39,963255,0.005162
7,921539,0.00509
61,855112,0.005641
59,776424,0.005336
63,586446,0.005484
49,486845,0.005476
35,461981,0.005305
79,450925,0.005553



10 самых редких item_id:


item_id,impressions,CTR
i64,u32,f64
75,17153,0.002099
21,16927,0.002186
40,16786,0.002621
70,15275,0.002291
74,15160,0.002045
25,15020,0.001931
20,12980,0.001772
10,11317,0.001325
54,8561,0.002687



Доля показов топ 1 item: 0.1093


In [54]:
print("Описательные статистики:")
display(df["propensity_score"].describe())


Описательные статистики:


statistic,value
str,f64
"""count""",1.2354503e7
"""null_count""",0.0
"""mean""",0.109425
"""std""",0.132652
"""min""",0.000001
"""25%""",0.023525
"""50%""",0.0666
"""75%""",0.14893
"""max""",0.95424


In [55]:
print("Диапазон времени:", df["ts"].min(), "-", df["ts"].max())
print("Длительность:", df["ts"].max() - df["ts"].min())

daily = (
    df.with_columns(pl.col("ts").dt.date().alias("date"))
      .group_by("date")
      .agg(pl.len().alias("n_impressions"),
           pl.col("click").mean().alias("ctr"),
           pl.col("item_id").n_unique().alias("n_unique_items"))
      .sort("date")
)
print("\nПоказы по дням:")
display(daily)

Диапазон времени: 2019-11-24 00:00:00.047846+00:00 - 2019-11-30 23:59:59.920907+00:00
Длительность: 6 days, 23:59:59.873061

Показы по дням:


date,n_impressions,ctr,n_unique_items
date,u32,f64,u32
2019-11-24,1870645,0.004643,80
2019-11-25,1479043,0.004707,80
2019-11-26,1611638,0.004757,80
2019-11-27,1881049,0.004908,80
2019-11-28,2069891,0.00461,80
2019-11-29,1799545,0.00466,80
2019-11-30,1642692,0.005255,80


In [56]:
hourly = (
    df.with_columns(pl.col("ts").dt.hour().alias("hour"))
      .group_by("hour")
      .agg(pl.len().alias("n_impressions"),
           pl.col("click").mean().alias("ctr"))
      .sort("hour")
)
print("Показы по часам:")
display(hourly)

Показы по часам:


hour,n_impressions,ctr
i8,u32,f64
0,452672,0.004774
1,502347,0.004519
2,531299,0.00371
3,699813,0.003815
4,574393,0.004332
…,…,…
19,95183,0.004696
20,129035,0.004572
21,227175,0.004538


In [57]:
click_rate = df["click"].mean()
item_stats = (
    df.group_by("item_id")
      .agg(pl.len().alias("impressions"))
      .sort("impressions", descending=True)
)
top1_share = item_stats["impressions"][0] / df.height

print(f"""
ВЫВОДЫ:
- Данные: {df.height:,} строк, {df.width} колонок.
- Click rate: {click_rate*100:.4f}% : клики очень редкие.
- Уникальных item_id: {item_stats.height}.
  Топ-1 item забирает {top1_share*100:.2f}% всех показов.
- Propensity: min={df['propensity_score'].min():.6f}, max={df['propensity_score'].max():.6f},
  mean={df['propensity_score'].mean():.6f}.
- Пропусков нет.
- Удалено 2,697 дубликатов 
""")


ВЫВОДЫ:
- Данные: 12,354,503 строк, 6 колонок.
- Click rate: 0.4784% : клики очень редкие.
- Уникальных item_id: 80.
  Топ-1 item забирает 10.93% всех показов.
- Propensity: min=0.000001, max=0.954240,
  mean=0.109425.
- Пропусков нет.
- Удалено 2,697 дубликатов 



In [88]:
PARQUET_PATH_RANDOM = r"random_all.parquet"
COLS_RANDOM = ["timestamp", "item_id", "position", "click", "propensity_score"]

In [77]:
df2 = (
    pl.read_parquet(PARQUET_PATH_RANDOM, columns=COLS_RANDOM)
      .with_columns(
          pl.col("timestamp").str.to_datetime(time_zone="UTC").alias("ts")
      )
)

print("Shape:", df2.shape)
print("Columns:", df2.columns)
print("Types:")
display(df2.schema)
print("\nNulls:")
display(df2.null_count())
print("\nHead:")
display(df2.head())

Shape: (1374327, 6)
Columns: ['timestamp', 'item_id', 'position', 'click', 'propensity_score', 'ts']
Types:


Schema([('timestamp', String),
        ('item_id', Int64),
        ('position', Int64),
        ('click', Int64),
        ('propensity_score', Float64),
        ('ts', Datetime(time_unit='us', time_zone='UTC'))])


Nulls:


timestamp,item_id,position,click,propensity_score,ts
u32,u32,u32,u32,u32,u32
0,0,0,0,0,0



Head:


timestamp,item_id,position,click,propensity_score,ts
str,i64,i64,i64,f64,"datetime[μs, UTC]"
"""2019-11-24 00:00:00.007365+00:…",21,1,0,0.0125,2019-11-24 00:00:00.007365 UTC
"""2019-11-24 00:00:00.015668+00:…",43,2,0,0.0125,2019-11-24 00:00:00.015668 UTC
"""2019-11-24 00:00:00.016438+00:…",62,3,0,0.0125,2019-11-24 00:00:00.016438 UTC
"""2019-11-24 00:00:01.856987+00:…",8,2,0,0.0125,2019-11-24 00:00:01.856987 UTC
"""2019-11-24 00:00:01.857185+00:…",45,1,0,0.0125,2019-11-24 00:00:01.857185 UTC


In [60]:
n_dups_random = df2.height - df2.unique().height
print(f"Полных дубликатов: {n_dups_random:,}")

Полных дубликатов: 165


In [61]:
dups_rows_random = (
    df2.group_by(COLS)
      .len()
      .filter(pl.col("len") > 1)
)
print(f"Уникальных строк, которые дублируются: {dups_rows_random.height}")

total_dup_rows_random = (dups_rows_random["len"] - 1).sum()
print(f"Лишних строк из-за дублирования: {total_dup_rows_random}")

Уникальных строк, которые дублируются: 146
Лишних строк из-за дублирования: 165


In [62]:
df_before_random = df2.height
df2 = df2.unique()
print(f"Было: {df_before_random:,}")
print(f"Стало: {df2.height:,}")
print(f"Удалено: {df_before_random - df2.height:,}")

Было: 1,374,327
Стало: 1,374,162
Удалено: 165


In [63]:
click_rate_random = df2["click"].mean()
print(f"Click rate: {click_rate_random:.6f} ({click_rate_random*100:.4f}%)")
print(f"Всего кликов: {df2['click'].sum():,} из {df2.height:,}")

print("\nРаспределение по позициям:")
display(
    df2.group_by("position")
      .agg(pl.len().alias("total_amount"), pl.col("click").mean().alias("CTR"))
      .sort("position")
)

Click rate: 0.003353 (0.3353%)
Всего кликов: 4,607 из 1,374,162

Распределение по позициям:


position,total_amount,CTR
i64,u32,f64
1,457950,0.003428
2,458109,0.003366
3,458103,0.003263


In [64]:
item_stats_random = (
    df2.group_by("item_id")
      .agg(pl.len().alias("impressions"),
           pl.col("click").mean().alias("CTR"))
      .sort("impressions", descending=True)
)

print("Уникальных item_id:", item_stats_random.height)
print("\nТоп 10 item_id по показам:")
display(item_stats_random.head(10))
print("\n10 самых редких item_id:")
display(item_stats_random.tail(10))

top1_share_random = item_stats_random["impressions"][0] / df2.height
print(f"\nДоля показов топ 1 item: {top1_share_random:.4f}")

Уникальных item_id: 80

Топ 10 item_id по показам:


item_id,impressions,CTR
i64,u32,f64
18,18178,0.003741
40,18015,0.001554
37,17786,0.004498
31,17776,0.003488
63,17746,0.005297
33,17733,0.002594
56,17700,0.001469
0,17657,0.002718
9,17650,0.002493



10 самых редких item_id:


item_id,impressions,CTR
i64,u32,f64
5,16594,0.002471
75,16594,0.002109
59,16593,0.004641
48,16562,0.005434
55,16546,0.003022
36,16543,0.003325
78,16492,0.00479
28,16395,0.002196
54,16330,0.001225



Доля показов топ 1 item: 0.0132


In [65]:
print("Описательные статистики:")
display(df2["propensity_score"].describe())

Описательные статистики:


statistic,value
str,f64
"""count""",1.374162e6
"""null_count""",0.0
"""mean""",0.0125
"""std""",1.8951e-20
"""min""",0.0125
"""25%""",0.0125
"""50%""",0.0125
"""75%""",0.0125
"""max""",0.0125


In [66]:
print("Диапазон времени:", df2["ts"].min(), "-", df2["ts"].max())
print("Длительность:", df2["ts"].max() - df2["ts"].min())

daily_random = (
    df2.with_columns(pl.col("ts").dt.date().alias("date"))
      .group_by("date")
      .agg(pl.len().alias("n_impressions"),
           pl.col("click").mean().alias("ctr"),
           pl.col("item_id").n_unique().alias("n_unique_items"))
      .sort("date")
)
print("\nПоказы по дням:")
display(daily_random)

Диапазон времени: 2019-11-24 00:00:00.007365+00:00 - 2019-11-30 23:59:59.408889+00:00
Длительность: 6 days, 23:59:59.401524

Показы по дням:


date,n_impressions,ctr,n_unique_items
date,u32,f64,u32
2019-11-24,206597,0.003282,80
2019-11-25,163972,0.003116,80
2019-11-26,179263,0.003671,80
2019-11-27,209723,0.002961,80
2019-11-28,231138,0.003513,80
2019-11-29,200234,0.003366,80
2019-11-30,183235,0.003564,80


In [67]:
hourly_random = (
    df2.with_columns(pl.col("ts").dt.hour().alias("hour"))
      .group_by("hour")
      .agg(pl.len().alias("n_impressions"),
           pl.col("click").mean().alias("ctr"))
      .sort("hour")
)
print("Показы по часам:")
display(hourly_random)

Показы по часам:


hour,n_impressions,ctr
i8,u32,f64
0,50973,0.003786
1,55983,0.003269
2,58855,0.003364
3,78770,0.002958
4,64163,0.003024
…,…,…
19,10556,0.004358
20,14562,0.004258
21,25258,0.003524


In [68]:
click_rate_random = df2["click"].mean()
item_stats_random = (
    df2.group_by("item_id")
      .agg(pl.len().alias("impressions"))
      .sort("impressions", descending=True)
)
top1_share_random = item_stats_random["impressions"][0] / df2.height

In [72]:
window_df = (
    df.with_columns(
        pl.col("ts").dt.truncate("10m").alias("window") 
    )
    .group_by("window")  
    .agg([
        pl.len().alias("n_impressions"),
        pl.col("item_id").n_unique().alias("n_unique_items"),
        pl.col("click").mean().alias("ctr"),
        pl.col("propensity_score").mean().alias("mean_propensity"),
        pl.col("propensity_score").std().alias("std_propensity"),
    ])
    .sort("window")
)

print(window_df)

shape: (1_008, 6)
┌───────────────────┬───────────────┬────────────────┬──────────┬─────────────────┬────────────────┐
│ window            ┆ n_impressions ┆ n_unique_items ┆ ctr      ┆ mean_propensity ┆ std_propensity │
│ ---               ┆ ---           ┆ ---            ┆ ---      ┆ ---             ┆ ---            │
│ datetime[μs, UTC] ┆ u32           ┆ u32            ┆ f64      ┆ f64             ┆ f64            │
╞═══════════════════╪═══════════════╪════════════════╪══════════╪═════════════════╪════════════════╡
│ 2019-11-24        ┆ 12224         ┆ 66             ┆ 0.005072 ┆ 0.073592        ┆ 0.065534       │
│ 00:00:00 UTC      ┆               ┆                ┆          ┆                 ┆                │
│ 2019-11-24        ┆ 11917         ┆ 70             ┆ 0.00428  ┆ 0.080298        ┆ 0.07684        │
│ 00:10:00 UTC      ┆               ┆                ┆          ┆                 ┆                │
│ 2019-11-24        ┆ 12613         ┆ 63             ┆ 0.005233 ┆ 0.06939

In [80]:
window_df_random = (
    df2.with_columns(
        pl.col("ts").dt.truncate("10m").alias("window") 
    )
    .group_by("window")  
    .agg([
        pl.len().alias("n_impressions"),
        pl.col("item_id").n_unique().alias("n_unique_items"),
        pl.col("click").mean().alias("ctr"),
        pl.col("propensity_score").mean().alias("mean_propensity"),
        pl.col("propensity_score").std().alias("std_propensity"),
    ])
    .sort("window")
)

print(window_df_random)

shape: (1_008, 6)
┌───────────────────┬───────────────┬────────────────┬──────────┬─────────────────┬────────────────┐
│ window            ┆ n_impressions ┆ n_unique_items ┆ ctr      ┆ mean_propensity ┆ std_propensity │
│ ---               ┆ ---           ┆ ---            ┆ ---      ┆ ---             ┆ ---            │
│ datetime[μs, UTC] ┆ u32           ┆ u32            ┆ f64      ┆ f64             ┆ f64            │
╞═══════════════════╪═══════════════╪════════════════╪══════════╪═════════════════╪════════════════╡
│ 2019-11-24        ┆ 1395          ┆ 79             ┆ 0.003584 ┆ 0.0125          ┆ 0.0            │
│ 00:00:00 UTC      ┆               ┆                ┆          ┆                 ┆                │
│ 2019-11-24        ┆ 1419          ┆ 75             ┆ 0.008457 ┆ 0.0125          ┆ 0.0            │
│ 00:10:00 UTC      ┆               ┆                ┆          ┆                 ┆                │
│ 2019-11-24        ┆ 1341          ┆ 77             ┆ 0.003729 ┆ 0.0125 

In [83]:
top1_bts = (
    df.with_columns(pl.col("ts").dt.truncate("10m").alias("window"))
      .group_by(["window", "item_id"])
      .len()
      .sort(["window", "len"], descending=[False, True])
      .group_by("window")
      .agg([
          pl.col("len").first().alias("top1_count"),
          pl.col("len").sum().alias("total"),
      ])
      .with_columns((pl.col("top1_count") / pl.col("total")).alias("top1_share"))
      .select(["window", "top1_share"])
)
window_df = window_df.join(top1_bts, on="window", how="left")

def entropy(counts):
    p = counts / counts.sum()
    p = p[p > 0]
    return -np.sum(p * np.log2(p))

item_dist_bts = (
    df.with_columns(pl.col("ts").dt.truncate("10m").alias("window"))
      .group_by(["window", "item_id"])
      .len()
)
entropy_bts = (
    item_dist_bts.group_by("window")
                 .agg(pl.col("len").alias("counts"))
                 .with_columns(
                     pl.col("counts").map_elements(
                         lambda x: entropy(np.array(x)), return_dtype=pl.Float64
                     ).alias("entropy")
                 )
                 .select(["window", "entropy"])
)
window_df = window_df.join(entropy_bts, on="window", how="left")

print(window_df.head(5))

shape: (5, 9)
┌────────────┬───────────┬───────────┬──────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ window     ┆ n_impress ┆ n_unique_ ┆ ctr      ┆ … ┆ std_prope ┆ top1_shar ┆ top1_shar ┆ entropy  │
│ ---        ┆ ions      ┆ items     ┆ ---      ┆   ┆ nsity     ┆ e         ┆ e_right   ┆ ---      │
│ datetime[μ ┆ ---       ┆ ---       ┆ f64      ┆   ┆ ---       ┆ ---       ┆ ---       ┆ f64      │
│ s, UTC]    ┆ u32       ┆ u32       ┆          ┆   ┆ f64       ┆ f64       ┆ f64       ┆          │
╞════════════╪═══════════╪═══════════╪══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 2019-11-24 ┆ 12224     ┆ 66        ┆ 0.005072 ┆ … ┆ 0.065534  ┆ 0.155105  ┆ 0.155105  ┆ 4.68309  │
│ 00:00:00   ┆           ┆           ┆          ┆   ┆           ┆           ┆           ┆          │
│ UTC        ┆           ┆           ┆          ┆   ┆           ┆           ┆           ┆          │
│ 2019-11-24 ┆ 11917     ┆ 70        ┆ 0.00428  ┆ … ┆ 0.07684   ┆ 0.180666  ┆

In [84]:
top1_random = (
    df2.with_columns(pl.col("ts").dt.truncate("10m").alias("window"))
       .group_by(["window", "item_id"])
       .len()
       .sort(["window", "len"], descending=[False, True])
       .group_by("window")
       .agg([
           pl.col("len").first().alias("top1_count"),
           pl.col("len").sum().alias("total"),
       ])
       .with_columns((pl.col("top1_count") / pl.col("total")).alias("top1_share"))
       .select(["window", "top1_share"])
)
window_df_random = window_df_random.join(top1_random, on="window", how="left")

item_dist_random = (
    df2.with_columns(pl.col("ts").dt.truncate("10m").alias("window"))
       .group_by(["window", "item_id"])
       .len()
)
entropy_random = (
    item_dist_random.group_by("window")
                    .agg(pl.col("len").alias("counts"))
                    .with_columns(
                        pl.col("counts").map_elements(
                            lambda x: entropy(np.array(x)), return_dtype=pl.Float64
                        ).alias("entropy")
                    )
                    .select(["window", "entropy"])
)
window_df_random = window_df_random.join(entropy_random, on="window", how="left")

print(window_df_random.head(5))

shape: (5, 8)
┌────────────┬────────────┬────────────┬──────────┬────────────┬────────────┬───────────┬──────────┐
│ window     ┆ n_impressi ┆ n_unique_i ┆ ctr      ┆ mean_prope ┆ std_propen ┆ top1_shar ┆ entropy  │
│ ---        ┆ ons        ┆ tems       ┆ ---      ┆ nsity      ┆ sity       ┆ e         ┆ ---      │
│ datetime[μ ┆ ---        ┆ ---        ┆ f64      ┆ ---        ┆ ---        ┆ ---       ┆ f64      │
│ s, UTC]    ┆ u32        ┆ u32        ┆          ┆ f64        ┆ f64        ┆ f64       ┆          │
╞════════════╪════════════╪════════════╪══════════╪════════════╪════════════╪═══════════╪══════════╡
│ 2019-11-24 ┆ 1395       ┆ 79         ┆ 0.003584 ┆ 0.0125     ┆ 0.0        ┆ 0.037993  ┆ 6.072955 │
│ 00:00:00   ┆            ┆            ┆          ┆            ┆            ┆           ┆          │
│ UTC        ┆            ┆            ┆          ┆            ┆            ┆           ┆          │
│ 2019-11-24 ┆ 1419       ┆ 75         ┆ 0.008457 ┆ 0.0125     ┆ 0.0        ┆

In [85]:
print("Сравнение")
print(f"bts:  n_unique_items mean = {window_df['n_unique_items'].mean():.2f}, top1_share mean = {window_df['top1_share'].mean():.4f}, mean_propensity = {window_df['mean_propensity'].mean():.4f}, entropy = {window_df['entropy'].mean():.2f}")
print(f"random: n_unique_items mean = {window_df_random['n_unique_items'].mean():.2f}, top1_share mean = {window_df_random['top1_share'].mean():.4f}, mean_propensity = {window_df_random['mean_propensity'].mean():.4f}, entropy = {window_df_random['entropy'].mean():.2f}")

Сравнение
bts:  n_unique_items mean = 59.36, top1_share mean = 0.1864, mean_propensity = 0.0963, entropy = 4.40
random: n_unique_items mean = 69.67, top1_share mean = 0.0416, mean_propensity = 0.0125, entropy = 5.84
